In [22]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta

# 物理熔炼大厂真实业务场景沙盒
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

print("=== 🪐 正在物理熔炼真实的千万级/黑产高压沙盒 ===")
np.random.seed(42)

# =====================================================================
# ⚔️ 场景一：量化交易 Tick 价格表（高密度微秒撞车 + 乱序）
# =====================================================================
# 生成 1 万条高频交易数据，高密度模拟同一毫秒内多笔交易乱序落盘
base_time = datetime(2026, 6, 13, 10, 0, 0)
# 故意让随机时间戳高度集中在少数毫秒内，造成大量撞车
random_ms = np.random.randint(0, 1000, size=10000) 
tick_times = [base_time + timedelta(milliseconds=int(ms)) for ms in random_ms]
tick_times_str = [t.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3] for t in tick_times]

df_ticks_raw = pd.DataFrame({
    'id': np.arange(1, 10001), # 自增物理落盘ID
    'ticker': ['AAPL'] * 10000,
    'tick_time': tick_times_str,
    'price': np.round(150.0 + np.cumsum(np.random.normal(0, 0.1, size=10000)), 2)
})
# 彻底打乱物理顺序，模拟分布式捞出的无序状态
df_ticks = df_ticks_raw.sample(frac=1).reset_index(drop=True)
df_ticks.to_sql('tick_prices', conn, if_exists='replace', index=False)
print(df_ticks.head())

# =====================================================================
# ⚔️ 场景二：电商风控行为日志表（黑产“幽灵爆刷”高密度噪声掩埋）
# =====================================================================
# 模拟黑产 U01 在抢券前，疯狂进行 1000 次点击和浏览来掩埋路径
u01_noise_types = np.random.choice(['click', 'browse'], size=1000)
u01_noise_times = [datetime(2026, 6, 13, 9, 0, 0) + timedelta(seconds=i) for i in range(1000)]

log_data = {
    'user_id': ['U01'] * 1000 + ['U01', 'U01', 'U02', 'U02'],
    'action_time': [t.strftime('%Y-%m-%d %H:%M:%S') for t in u01_noise_times] + [
        '2026-06-13 09:05:00', # U01 抢券1
        '2026-06-13 09:15:00', # U01 抢券2
        '2026-06-13 14:00:00', # U02 纯路人冷启动
        '2026-06-13 14:05:00'
    ],
    'action_type': list(u01_noise_types) + ['coupon_grab', 'coupon_grab', 'browse', 'click']
}
df_logs_raw = pd.DataFrame(log_data)
# 彻底乱序
df_logs = df_logs_raw.sample(frac=1).reset_index(drop=True)
df_logs.to_sql('user_behavior_logs', conn, if_exists='replace', index=False)


# =====================================================================
# ⚔️ 场景三：AB测试活跃用户主表（严格控制为 102 行）
# =====================================================================
df_users = pd.DataFrame({
    'user_id': [f'USER_{i:03d}' for i in range(1, 103)]
})
df_users.to_sql('active_users', conn, if_exists='replace', index=False)

print("✅ 真实业务场景沙盒熔炼完成！你可以开始用代码迎战面试官了。\n")

=== 🪐 正在物理熔炼真实的千万级/黑产高压沙盒 ===
     id ticker                tick_time   price
0  3212   AAPL  2026-06-13 10:00:00.737  151.32
1  1032   AAPL  2026-06-13 10:00:00.708  146.50
2  7239   AAPL  2026-06-13 10:00:00.057  146.36
3  9160   AAPL  2026-06-13 10:00:00.097  142.73
4   809   AAPL  2026-06-13 10:00:00.947  147.42
✅ 真实业务场景沙盒熔炼完成！你可以开始用代码迎战面试官了。



## ⚔️ 第一题：量化交易宇宙 —— “闪崩动量”特征提取（考核：`LAG` + `ROWS` 边界）

### 📊 业务沙盒：

高频交易系统里有一张股票Tick级别订单流表 `tick_prices`：

- `ticker` (股票代码)
    
- `tick_time` (精确到毫秒的时间戳)
    
- `price` (当前成交价)
    

### 🚨 面试官断头台提问：

“请为机器学习模型加工一个特征 `price_drop_3ticks`。这个特征的逻辑是：**用当前这一行的价格，减去过去（不含当前行）连续第 3 个物理格子那一行的价格。** 并且请回答我：如果高并发导致多条 tick 数据在同一毫秒内涌入（时间戳完全一样），你的 SQL 和 Pandas 应该如何做，才能绝对保证特征计算的确定性，不发生逻辑混乱？”





In [41]:
import os
import pandas as pd

# =====================================================================
# 🛡️ 1. 自适应工程路径安全闭环（逆向爬树算法）
# =====================================================================
current_dir = os.getcwd()
while os.path.basename(current_dir) != "data-science-labs":
    parent_dir = os.path.dirname(current_dir) # 计算父目录
    if parent_dir == current_dir: # 如果父目录和当前目录一致，结束循环
        break
    current_dir = parent_dir # 否则就继续后退
output_path = os.path.join(current_dir, "raw_data", "cleaned_tick_drops.csv")

if os.path.exists(output_path):
    os.remove(output_path)

# =====================================================================
# 🚀 2. 终极投产版 SQL：把所有重体力活、窗口算法、排毒防御全部下沉到数仓
# =====================================================================
perfect_production_sql = """
WITH src_cleaned AS (
    SELECT 
        ticker,
        tick_time,
        price,
        -- 【三级火箭强排序】在数仓底层直接拉直高并发撞车，分配绝对唯一的序号
        ROW_NUMBER() OVER(
            PARTITION BY ticker 
            ORDER BY tick_time ASC, id ASC
        ) AS sequence_id
    FROM tick_prices
    -- 刚性分区裁剪：卡死 2026-06-13，干掉全表扫描；物理排毒干掉脏数据
    WHERE tick_time >= '2026-06-13 00:00:00' 
      AND tick_time < '2026-06-14 00:00:00'
      AND price IS NOT NULL  
      AND price > 0          
),
feature_calculated AS (
    SELECT 
        ticker,
        tick_time,
        price,
        -- 严格依赖唯一序号排序，LAG(3) 在数仓全局闭宇宙中完美滑窗，绝无边界断裂
        LAG(price, 3) OVER(
            PARTITION BY ticker 
            ORDER BY tick_time ASC, sequence_id ASC
        ) AS last_3th_price
    FROM src_cleaned
)
-- 统计学防漏拦截：只把 100% 纯净的特征成品输送给 Python 管道
SELECT 
    ticker,
    tick_time,
    price,
    last_3th_price,
    (price - last_3th_price) AS price_drop_3ticks
FROM feature_calculated
WHERE last_3th_price IS NOT NULL;
"""

# =====================================================================
# 💾 3. Pandas 流式低水位消费流水线
# =====================================================================
CHUNK_SIZE = 100000  # 每次只放 10 万条特征成品进 Python 内存，卡死内存曲线
total_records = 0

try:
    # 建立流式管道生成器
    query_generator = pd.read_sql_query(perfect_production_sql, conn, chunksize=CHUNK_SIZE)
    print("【工业级双轨合流引擎启动】远端数仓开始疯狂倾泻纯净特征流...\n")
    
    for i, feature_chunk in enumerate(query_generator):
        # 🔔 此时的 feature_chunk 已经完全由 SQL 算好了！
        # Pandas 引擎处于零负载状态，无需任何 sort_values, groupby, shift 或 datetime 转换！
        
        # 确定性流式增量落盘
        if i == 0:
            feature_chunk.to_csv(output_path, mode='w', index=False)
        else:
            feature_chunk.to_csv(output_path, mode='a', header=False, index=False)
            
        current_rows = len(feature_chunk)
        total_records += current_rows
        # 🔍 动态滚动对账：同时喷出“当前批次行数”与“系统累计总行数”
    # 🔍 动态滚动对账：同时喷出“当前批次行数”与“系统累计总行数”
    print(
    f"-> 传送带无缝接单批次 {i+1}: Python 零负载，"
    f"本批落盘 {current_rows} 条。累计安全放行总盘子: {total_records} 条特征。"
)

    print(f"\n🎉 【完美通关】全量高纯度特征已安全持久化至: {output_path}")

except Exception as e:
    print(f"🚨 【重大生产事故】投产管道运行崩溃: {e}")

【工业级双轨合流引擎启动】远端数仓开始疯狂倾泻纯净特征流...

-> 传送带无缝接单批次 1: Python 零负载，本批落盘 9997 条。累计安全放行总盘子: 9997 条特征。

🎉 【完美通关】全量高纯度特征已安全持久化至: c:\projects\data-science-labs\raw_data\cleaned_tick_drops.csv


In [42]:
import pandas as pd
try:
    # 读取物理文件
    df_result = pd.read_csv(output_path)
    
    # 打印全局维度对账
    print("==========================================================")
    print(f"📊 物理文件读取成功！")
    print(f"表格实际总行数: {df_result.shape[0]} 行")
    print(f"表格实际总列数: {df_result.shape[1]} 列")
    print("==========================================================")
    
    # 优雅展示前 10 行数据
    print("\n🔬 【数据前 10 行高纯度特征真实排布】")
    # to_string(index=False) 可以去掉左侧难看的 Pandas 自增索引，让输出像数仓一样干净
    print(df_result.head(10).to_string(index=False))

except FileNotFoundError:
    print(f"❌ 警告：在当前工作目录下未找到文件 {output_path}，请检查路径。")

📊 物理文件读取成功！
表格实际总行数: 9997 行
表格实际总列数: 5 列

🔬 【数据前 10 行高纯度特征真实排布】
ticker               tick_time  price  last_3th_price  price_drop_3ticks
  AAPL 2026-06-13 10:00:00.000 148.14          145.89               2.25
  AAPL 2026-06-13 10:00:00.000 150.16          147.26               2.90
  AAPL 2026-06-13 10:00:00.000 150.68          147.74               2.94
  AAPL 2026-06-13 10:00:00.000 150.53          148.14               2.39
  AAPL 2026-06-13 10:00:00.000 149.50          150.16              -0.66
  AAPL 2026-06-13 10:00:00.000 146.60          150.68              -4.08
  AAPL 2026-06-13 10:00:00.000 147.81          150.53              -2.72
  AAPL 2026-06-13 10:00:00.000 145.29          149.50              -4.21
  AAPL 2026-06-13 10:00:00.000 139.81          146.60              -6.79
  AAPL 2026-06-13 10:00:00.000 141.97          147.81              -5.84


## ⚔️ 第二题：电商风控宇宙 —— “幽灵爆刷”路径穿透（考核：`LAST_VALUE` + `CASE WHEN` + `1 PRECEDING`）

### 📊 业务沙盒：

用户行为风控日志表 `user_behavior_logs`：

- `user_id` (用户唯一标识)
    
- `action_time` (操作时间戳)
    
- `action_type` (行为类型：`click` 页面点击, `browse` 浏览商品, `coupon_grab` 抢代金券)
    

### 🚨 面试官断头台提问：

“黑产团伙在发起攻击前，往往会先进行疯狂的点击和浏览伪装。运营团队要求你提取一个特征 `last_coupon_time`：**计算当前行为发生之前（不含当前行），该用户上一次发生『抢代金券（coupon_grab）』动作的绝对时间戳。** 请分别用 **纯 SQL（必须包含显式窗口声明）** 和 **Pandas 链式调用（必须包含错位与广播算子）** 写出无懈可击的对账代码。如果这个用户在过去从来没有抢过券，该特征应当安全回填为什么默认值？”

## ⚔️ 第三题：大厂流量治理 —— “灰度流量”AB测试分流（考核：`NTILE` 溢出边界）

### 📊 业务沙盒：

用户留存主表 `active_users` 共有 **102 个** 活跃用户。

### 🚨 面试官断头台提问：

“算法团队需要在线上紧急开启一个 10 桶灰度测试（AB Test），要求你用 `NTILE(10)` 将这 102 个用户分配到 1 到 10 号试验桶中。 请闭眼心算，不需要写代码，直接回答我三个微观边界：

1. 1 号桶到 10 号桶，各自会分到多少个用户？
    
2. 计算机底层的『动态配额递减算法』是如何处理这多出来的 2 个余数用户的？
    
3. 为什么大厂在做流量分发时，更倾向于用 NTILE 而不是简单的随机数？”